# 레슨 08 — 시각화 기초 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 표를 차트로 바꾸는 것이 아니라, 분석 질문에 맞는 차트를 고르고 숫자와 결론을 연결하는 것이다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/08/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 시각화 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/visual_insights.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("앞 5행:")
print(df.head())
print("뒤 3행:")
print(df.tail(3))
print("결측치:")
print(df.isna().sum())

df["date"] = pd.to_datetime(df["date"])
print("날짜 범위:", df["date"].min().date(), "~", df["date"].max().date())

### 왜 이 코드가 정답인지

시각화는 열의 의미와 타입을 확인한 뒤 시작해야 한다. `shape`, `columns`, `dtypes`, `isna()` 를 확인하면 어떤 열을 x축, y축, 색상 구분에 사용할 수 있는지 판단할 수 있다. `date` 를 날짜형으로 바꾸면 뒤 문제에서 월별 집계와 선 그래프를 안정적으로 만들 수 있다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 행 수 | 540 |
| 열 수 | 7 |
| 결측치 | 0개 |

---

## 문제 2 정답 — 분석용 기본 열 만들기

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")
df["weekday"] = df["date"].dt.dayofweek
df["roi"] = df["revenue"] / df["ad_spend"]

print(df[["date", "month", "weekday", "channel", "category", "revenue", "ad_spend", "roi"]].head())
print("월 목록:", sorted(df["month"].unique()))
print("요일 번호:", sorted(df["weekday"].unique()))

### 왜 이 코드가 정답인지

`month` 는 일별 데이터를 월별로 줄이기 위한 기준이고, `weekday` 는 요일 패턴을 볼 때 사용할 수 있는 기준이다. `roi` 는 광고비 대비 매출을 나타내므로 단순 매출 규모와 효율을 분리해서 볼 수 있다. 이 세 열을 먼저 만들어 두면 뒤 문제에서 같은 계산을 반복하지 않아도 된다.

**예상 핵심값**

```text
월: 2025-01 ~ 2025-06
요일 번호: 0 ~ 6
```

---

## 문제 3 정답 — 일별 매출 선 그래프 준비

In [ ]:
daily = df.groupby("date", as_index=False).agg(
    orders=("orders", "sum"),
    revenue=("revenue", "sum"),
    ad_spend=("ad_spend", "sum"),
)

print(daily.head())

fig, ax = plt.subplots(figsize=(8, 3))
sns.lineplot(data=daily, x="date", y="revenue", ax=ax)
ax.set_title("Daily revenue trend")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue")
fig.tight_layout()
plt.close(fig)

top_day = daily.loc[daily["revenue"].idxmax()]
print("매출 최고 날짜:", top_day["date"].date(), "매출:", int(top_day["revenue"]))

### 왜 이 코드가 정답인지

한 날짜에 여러 채널과 카테고리 행이 있으므로 일별 흐름을 보려면 먼저 날짜별 합계를 만들어야 한다. 선 그래프는 시간 순서가 있는 값을 연결해서 증가와 감소를 보여주기에 적합하다. 최고 날짜를 함께 출력하면 차트를 보지 못하는 환경에서도 핵심 숫자를 확인할 수 있다.

**예상 핵심값**

```text
daily 행 수: 180일
```

---

## 문제 4 정답 — 채널별 성과 막대 그래프

In [ ]:
channel_summary = (
    df.groupby("channel")
    .agg(
        orders=("orders", "sum"),
        revenue=("revenue", "sum"),
        ad_spend=("ad_spend", "sum"),
        conversion_rate=("conversion_rate", "mean"),
        roi=("roi", "mean"),
    )
    .sort_values("revenue", ascending=False)
)

print(channel_summary)

fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(data=channel_summary.reset_index(), x="channel", y="revenue", ax=ax)
ax.set_title("Revenue by channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Revenue")
fig.tight_layout()
plt.close(fig)

print("매출 1위 채널:", channel_summary["revenue"].idxmax())
print("ROI 1위 채널:", channel_summary["roi"].idxmax())

### 왜 이 코드가 정답인지

채널별 성과는 주문, 매출, 광고비처럼 더해야 하는 값과 전환율, ROI처럼 평균으로 봐야 하는 값을 구분해야 한다. `groupby().agg()` 는 열마다 다른 집계 기준을 지정할 수 있어 이 문제에 적합하다. 막대 그래프는 채널처럼 순서가 없는 범주를 비교할 때 가장 읽기 쉽다.

**채점 기준**

| 확인 항목 | 기준 |
|---|---|
| 집계 | 채널별로 묶었는가 |
| 정렬 | 매출 기준 내림차순인가 |
| 해석 | 매출 1위와 효율 1위를 구분했는가 |

---

## 문제 5 정답 — 카테고리별 성과 막대 그래프

In [ ]:
category_summary = (
    df.groupby("category")
    .agg(
        orders=("orders", "sum"),
        revenue=("revenue", "sum"),
        conversion_rate=("conversion_rate", "mean"),
        roi=("roi", "mean"),
    )
    .sort_values("revenue", ascending=False)
)

print(category_summary)

fig, ax = plt.subplots(figsize=(6, 3))
sns.barplot(data=category_summary.reset_index(), x="category", y="revenue", ax=ax)
ax.set_title("Revenue by category")
ax.set_xlabel("Category")
ax.set_ylabel("Revenue")
fig.tight_layout()
plt.close(fig)

print("매출 1위 카테고리:", category_summary["revenue"].idxmax())
print("전환율 1위 카테고리:", category_summary["conversion_rate"].idxmax())

### 왜 이 코드가 정답인지

문제 4와 같은 집계 패턴을 카테고리에 적용하면 채널 비교와 상품군 비교를 같은 기준으로 진행할 수 있다. 매출 1위와 전환율 1위가 다를 수 있으므로 두 기준을 따로 출력한다. 이렇게 해야 "많이 팔린 카테고리"와 "방문이 구매로 잘 바뀐 카테고리"를 구분할 수 있다.

**예상 확인 포인트**

```text
category_summary 의 index 는 category
orders, revenue, conversion_rate, roi 열 포함
```

---

## 문제 6 정답 — 월별 채널 매출 추이

In [ ]:
monthly_channel = (
    df.groupby(["month", "channel"], as_index=False)
    .agg(revenue=("revenue", "sum"), orders=("orders", "sum"))
)

print(monthly_channel.head(10))

fig, ax = plt.subplots(figsize=(8, 3))
sns.lineplot(data=monthly_channel, x="month", y="revenue", hue="channel", marker="o", ax=ax)
ax.set_title("Monthly revenue by channel")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
fig.tight_layout()
plt.close(fig)

monthly_total = df.groupby("month", as_index=False)["revenue"].sum()
top_month = monthly_total.loc[monthly_total["revenue"].idxmax()]
print("전체 매출 1위 월:", top_month["month"], int(top_month["revenue"]))

### 왜 이 코드가 정답인지

월과 채널을 함께 묶으면 시간 흐름과 그룹 차이를 동시에 볼 수 있다. `hue="channel"` 은 같은 x축에서 채널별 선을 나누어 보여주므로 비교가 쉽다. 전체 매출 1위 월을 별도로 계산하면 채널별 추이와 전체 추이를 혼동하지 않게 된다.

**수업 중 확인 질문**

```text
월별 전체 매출 1위와 특정 채널의 1위 월이 항상 같은가?
```

---

## 문제 7 정답 — 주문 수와 매출 분포 보기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
sns.histplot(data=df, x="orders", bins=20, ax=axes[0])
axes[0].set_title("Orders distribution")
sns.histplot(data=df, x="revenue", bins=20, ax=axes[1])
axes[1].set_title("Revenue distribution")
fig.tight_layout()
plt.close(fig)

distribution_summary = df[["orders", "revenue"]].agg(["mean", "median", "min", "max"])
print(distribution_summary)
print("orders 평균-중앙값 차이:", distribution_summary.loc["mean", "orders"] - distribution_summary.loc["median", "orders"])
print("revenue 평균-중앙값 차이:", distribution_summary.loc["mean", "revenue"] - distribution_summary.loc["median", "revenue"])

### 왜 이 코드가 정답인지

히스토그램은 값이 어느 구간에 많이 몰려 있는지 보여준다. 평균과 중앙값을 함께 보면 일부 큰 값이 분포를 끌어올리는지 판단할 수 있다. `orders` 와 `revenue` 를 나란히 그리면 주문 수 분포와 매출 분포가 비슷한지 비교할 수 있다.

**해석 기준**

| 상황 | 해석 |
|---|---|
| 평균이 중앙값보다 큼 | 큰 값이 오른쪽 꼬리를 만들 가능성 |
| 평균과 중앙값이 비슷함 | 비교적 균형 잡힌 분포 |

---

## 문제 8 정답 — 광고비와 매출 관계 산점도

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.scatterplot(data=df, x="ad_spend", y="revenue", hue="channel", alpha=0.75, ax=ax)
ax.set_title("Ad spend vs revenue")
ax.set_xlabel("Ad spend")
ax.set_ylabel("Revenue")
fig.tight_layout()
plt.close(fig)

corr_cols = ["ad_spend", "revenue", "orders", "conversion_rate"]
corr = df[corr_cols].corr()
print(corr.round(3))

revenue_corr = corr["revenue"].drop("revenue").abs().sort_values(ascending=False)
print("매출과 가장 함께 움직인 지표:", revenue_corr.index[0], f"{revenue_corr.iloc[0]:.3f}")

### 왜 이 코드가 정답인지

산점도는 두 숫자 열의 관계를 직접 보여준다. 광고비가 증가할수록 매출도 증가하는지, 채널별로 점의 위치가 다른지 확인할 수 있다. 상관계수를 함께 계산하면 눈으로 본 관계를 숫자로 검증할 수 있지만, 상관이 높다고 해서 광고비가 매출의 유일한 원인이라고 말할 수는 없다.

**주의할 점**

```text
상관관계는 같이 움직이는 정도이고, 원인과 결과를 확정하지 않는다.
```

---

## 문제 9 정답 — 전환율 박스플롯

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sns.boxplot(data=df, x="channel", y="conversion_rate", ax=axes[0])
axes[0].set_title("Conversion rate by channel")
sns.boxplot(data=df, x="category", y="conversion_rate", ax=axes[1])
axes[1].set_title("Conversion rate by category")
fig.tight_layout()
plt.close(fig)

channel_conversion_median = df.groupby("channel")["conversion_rate"].median().sort_values(ascending=False)
category_conversion_median = df.groupby("category")["conversion_rate"].median().sort_values(ascending=False)

print("채널별 전환율 중앙값:")
print(channel_conversion_median)
print("카테고리별 전환율 중앙값:")
print(category_conversion_median)
print("전환율 중앙값 1위 채널:", channel_conversion_median.index[0])

### 왜 이 코드가 정답인지

박스플롯은 그룹별 중앙값, 사분위 범위, 이상치를 한 번에 보여준다. 전환율은 평균만 보면 일부 큰 값에 영향을 받을 수 있으므로 중앙값도 확인해야 한다. 채널과 카테고리를 각각 그리면 어떤 기준에서 전환율 차이가 더 뚜렷한지 비교할 수 있다.

**수업 중 피드백 포인트**

```text
학생이 "높다"라고 말하면 평균인지 중앙값인지 다시 묻는다.
```

---

## 문제 10 정답 — 숫자 열 상관관계 히트맵

In [ ]:
numeric_cols = ["orders", "revenue", "ad_spend", "conversion_rate", "roi"]
numeric_corr = df[numeric_cols].corr()

print(numeric_corr.round(2))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(numeric_corr, annot=True, cmap="Blues", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation heatmap")
fig.tight_layout()
plt.close(fig)

corr_pairs = numeric_corr.where(~np.eye(len(numeric_corr), dtype=bool)).stack()
strong_pair = corr_pairs.abs().idxmax()
weak_pair = corr_pairs.abs().idxmin()

print("가장 큰 상관 조합:", strong_pair, f"{corr_pairs.loc[strong_pair]:.3f}")
print("가장 작은 상관 조합:", weak_pair, f"{corr_pairs.loc[weak_pair]:.3f}")

### 왜 이 코드가 정답인지

상관계수 표는 숫자 열 사이의 관계를 정리하고, 히트맵은 그 관계를 색으로 빠르게 읽게 해준다. `np.eye` 로 대각선을 제외해야 자기 자신과의 상관계수 1.0이 항상 1위로 잡히는 문제를 피할 수 있다. 가장 큰 조합과 가장 작은 조합을 출력하면 색상 해석을 숫자로 확인할 수 있다.

**채점 기준**

| 확인 항목 | 기준 |
|---|---|
| 숫자 열 선택 | 범주 열을 제외했는가 |
| 대각선 제외 | 자기 자신과의 상관을 뺐는가 |
| 해석 | 큰 조합과 작은 조합을 구분했는가 |

---

## 문제 11 정답 — 카테고리 × 채널 피벗 히트맵

In [ ]:
category_channel_pivot = pd.pivot_table(
    df,
    values="revenue",
    index="category",
    columns="channel",
    aggfunc="sum",
    fill_value=0,
)

print(category_channel_pivot)

fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(category_channel_pivot, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax)
ax.set_title("Revenue pivot: category x channel")
fig.tight_layout()
plt.close(fig)

stacked = category_channel_pivot.stack()
print("매출 최대 조합:", stacked.idxmax(), int(stacked.max()))
print("매출 최소 조합:", stacked.idxmin(), int(stacked.min()))

### 왜 이 코드가 정답인지

피벗 테이블은 두 범주를 축으로 두고 숫자 값을 요약할 때 적합하다. `category` 와 `channel` 을 함께 보면 어떤 상품군이 어떤 채널에서 강한지 알 수 있다. 히트맵은 표의 큰 값과 작은 값을 빠르게 찾도록 돕지만, 최종 결론에는 `idxmax()` 와 `idxmin()` 으로 확인한 실제 조합을 사용해야 한다.

**예상 확인 포인트**

```text
행: category
열: channel
값: revenue 합계
```

---

## 문제 12 정답 — 4분할 대시보드 만들기

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

sns.lineplot(data=daily, x="date", y="revenue", ax=axes[0, 0])
axes[0, 0].set_title("Daily revenue")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Revenue")

sns.barplot(data=channel_summary.reset_index(), x="channel", y="revenue", ax=axes[0, 1])
axes[0, 1].set_title("Revenue by channel")
axes[0, 1].set_xlabel("Channel")
axes[0, 1].set_ylabel("Revenue")

sns.histplot(data=df, x="orders", bins=20, ax=axes[1, 0])
axes[1, 0].set_title("Orders distribution")
axes[1, 0].set_xlabel("Orders")

sns.scatterplot(data=df, x="ad_spend", y="revenue", hue="channel", alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title("Ad spend vs revenue")
axes[1, 1].set_xlabel("Ad spend")
axes[1, 1].set_ylabel("Revenue")

fig.suptitle("Visual insight dashboard", fontsize=14)
fig.tight_layout()
plt.close(fig)

print("대시보드 차트 4개 생성 완료")

### 왜 이 코드가 정답인지

대시보드는 여러 차트를 한 화면에 배치해 시간 흐름, 범주 비교, 분포, 관계를 동시에 보게 한다. `plt.subplots(2, 2)` 는 네 개 영역을 명확히 만들고, 각 축에 다른 seaborn 차트를 넣을 수 있다. 제목과 축 라벨을 붙이면 학생이 차트만 보고도 질문을 이해할 수 있다.

**교사용 확인 질문**

```text
이 4개 차트가 서로 다른 질문에 답하는가?
```

---

## 문제 13 정답 — 차트에 사용할 핵심 문장 만들기

In [ ]:
top_channel = channel_summary["revenue"].idxmax()
top_channel_revenue = channel_summary.loc[top_channel, "revenue"]

top_category = category_summary["revenue"].idxmax()
top_category_revenue = category_summary.loc[top_category, "revenue"]

top_conversion_channel = channel_conversion_median.index[0]
top_conversion_value = channel_conversion_median.iloc[0]

top_month = monthly_total.loc[monthly_total["revenue"].idxmax()]

messages = [
    f"매출 1위 채널은 {top_channel}이며 총매출은 {top_channel_revenue:,.0f}입니다.",
    f"매출 1위 카테고리는 {top_category}이며 총매출은 {top_category_revenue:,.0f}입니다.",
    f"전환율 중앙값이 가장 높은 채널은 {top_conversion_channel}입니다.",
    f"월별 전체 매출은 {top_month['month']}에 가장 높았습니다.",
]

for message in messages:
    print(message)

### 왜 이 코드가 정답인지

발표용 문장은 차트에서 읽은 핵심 숫자를 짧게 고정하는 역할을 한다. 이 문제는 새 계산보다 앞에서 만든 요약표를 정확히 재사용하는 것이 중요하다. 같은 변수에서 문장을 만들면 차트와 결론이 서로 다른 값을 말하는 실수를 줄일 수 있다.

**좋은 문장 조건**

| 요소 | 설명 |
|---|---|
| 대상 | 채널, 카테고리, 월처럼 비교 대상이 명확함 |
| 숫자 | 매출, 전환율 같은 근거가 포함됨 |
| 방향 | 높다, 낮다, 우선 본다 같은 해석이 있음 |

---

## 문제 14 정답 — 차트 선택 계획표 작성

In [ ]:
chart_plan = pd.DataFrame(
    [
        {"question": "일별 매출은 어떻게 변했는가?", "chart": "line", "x": "date", "y": "revenue", "reason": "시간 흐름 비교"},
        {"question": "채널별 매출 규모는 어떻게 다른가?", "chart": "bar", "x": "channel", "y": "revenue", "reason": "범주별 합계 비교"},
        {"question": "주문 수는 어떤 구간에 몰려 있는가?", "chart": "histogram", "x": "orders", "y": "", "reason": "숫자 분포 확인"},
        {"question": "광고비와 매출은 함께 움직이는가?", "chart": "scatter", "x": "ad_spend", "y": "revenue", "reason": "두 숫자 관계 확인"},
        {"question": "채널별 전환율 분포는 어떤가?", "chart": "boxplot", "x": "channel", "y": "conversion_rate", "reason": "그룹별 중앙값과 퍼짐 확인"},
        {"question": "카테고리와 채널 조합은 어디가 강한가?", "chart": "heatmap", "x": "channel", "y": "category", "reason": "두 범주 조합 비교"},
    ]
)

print(chart_plan)
print("차트 종류 수:", chart_plan["chart"].nunique())
print("중복 질문 수:", chart_plan["question"].duplicated().sum())

### 왜 이 코드가 정답인지

차트 계획표는 분석 질문, 차트 종류, 축, 선택 이유를 한곳에 묶는다. 이렇게 하면 학생이 그래프를 많이 그리는 데서 멈추지 않고 각 차트가 어떤 질문에 답하는지 점검할 수 있다. `nunique()` 와 `duplicated()` 로 차트 종류와 질문 중복 여부를 확인하면 발표 구성이 한쪽으로 치우치는 것을 막을 수 있다.

**교사용 피드백 기준**

```text
차트 종류가 다양해도 질문이 중복되면 발표 자료로는 약하다.
```

---

## 문제 15 정답 — 시각화 리포트 결론

In [ ]:
print("채널별 성과:")
print(channel_summary)
print()

print("카테고리별 성과:")
print(category_summary)
print()

top_month = monthly_total.loc[monthly_total["revenue"].idxmax()]
print("월별 매출 1위:", top_month["month"], int(top_month["revenue"]))

strong_pair = corr_pairs.abs().idxmax()
print("상관관계 핵심 조합:", strong_pair, f"{corr_pairs.loc[strong_pair]:.3f}")

print("결론 초안:")
print(f"- 전체 매출은 {top_month['month']}에 가장 높았다.")
print(f"- 채널 기준 매출 1위는 {top_channel}, 카테고리 기준 매출 1위는 {top_category}이다.")
print(f"- 전환율 중앙값은 {top_conversion_channel} 채널이 가장 높다.")
print("- 다음 분석에서는 매출 규모와 ROI를 분리해서 캠페인 우선순위를 비교한다.")

### 왜 이 코드가 정답인지

마지막 결론은 새 계산을 많이 하는 단계가 아니라 앞에서 만든 핵심 표를 다시 확인하는 단계다. 채널, 카테고리, 월, 상관관계 값을 함께 출력하면 리포트가 여러 차트와 연결된다. 결론 문장에 "다음 분석 행동"까지 포함하면 단순 관찰을 운영 판단으로 바꿀 수 있다.

**모범 결론 예시**

```text
전체 매출은 특정 월에 집중되어 있으므로 월별 캠페인 일정과 함께 확인해야 한다.
채널별 매출 1위와 ROI 1위가 다르다면 규모와 효율을 분리해 판단한다.
카테고리별 매출 차이는 재고나 콘텐츠 배치 우선순위를 정하는 기준이 된다.
다음 분석에서는 전환율이 높은 조합에 광고비를 더 투입했을 때 매출이 함께 증가하는지 확인한다.
```

---

## 채점 포인트

| 항목 | 확인 기준 |
|---|---|
| 파일 로드 | `visual_insights.csv` 를 `DATA_BASE` 기준으로 읽음 |
| 구조 확인 | `shape`, `columns`, `dtypes`, 결측치 확인 |
| 시각화 | 선, 막대, 히스토그램, 산점도, 박스플롯, 히트맵 중 5종 이상 |
| 집계 | 채널, 카테고리, 월 기준 요약표를 만듦 |
| 결론 | 차트의 숫자와 모순 없는 행동 제안 포함 |

## 흔한 오답

- 날짜형 변환 없이 월별 그래프를 만들려고 한다.
- 합계를 써야 할 열과 평균을 써야 할 열을 구분하지 않는다.
- 막대 그래프에 원본 행을 바로 넣어 집계 기준이 불명확하다.
- 히트맵 색만 보고 실제 숫자를 확인하지 않는다.
- 결론에 차트 종류만 나열하고 비교 대상이나 다음 행동이 없다.

## 교사용 상세 피드백 가이드

학생 답안을 볼 때는 차트가 예쁜지보다 질문에 맞는지 먼저 본다. 다음 순서로 확인한다.

1. **같은 질문을 풀었는가**: 시각화 문제는 차트 종류보다 질문과 축 선택이 더 중요하다. 시간 흐름 문제에 막대 그래프를 써도 틀렸다고 단정할 필요는 없지만, 왜 그 차트를 골랐는지 설명하게 한다.
2. **집계 기준이 맞는가**: 채널별 매출은 합계, 전환율은 평균 또는 중앙값이 자연스럽다. 모든 열을 무조건 평균 내면 매출 규모 해석이 약해진다.
3. **차트 제목과 축 라벨이 있는가**: 제목이 없으면 학생이 무엇을 비교하는지 놓치기 쉽다. 축 라벨은 특히 광고비, 매출처럼 단위가 큰 숫자에서 중요하다.
4. **결론이 숫자와 맞는가**: 차트에서 본 1위와 결론의 1위가 다르면 시각화 리포트로는 실패다.

## 부분 점수 기준

| 상황 | 처리 |
|---|---|
| 파일 로드와 구조 확인만 성공 | 문제 1 일부 통과 |
| 차트는 있으나 집계 기준이 불명확 | 코드 실행 인정, 분석 기준 수정 |
| 차트 제목이 없음 | 통과 가능, 발표 품질 피드백 |
| 상관관계를 인과관계로 표현 | 해석 오류로 수정 필요 |
| 결론이 비어 있음 | 코드 점수 인정, 결론 재제출 |

## 보너스 확장 아이디어

빠른 학생에게는 같은 데이터를 다른 질문으로 다시 그리게 한다. 예를 들어 주중과 주말의 전환율 차이, 월별 채널 점유율, 카테고리별 ROI 순위, 광고비 구간별 매출 분포를 추가할 수 있다. 단, 차트를 늘릴수록 발표가 산만해지므로 최종 리포트에는 질문이 다른 차트만 남기게 한다.

## 수업 중 피드백 문장 예시

- “이 차트는 어떤 질문에 답하나요?”
- “매출 합계와 전환율 평균을 같은 기준으로 해석하면 안 됩니다.”
- “색이 예쁜 것보다 축과 제목이 정확한지가 먼저입니다.”
- “상관관계를 원인처럼 말하지 말고 같이 움직이는 정도로 표현합시다.”

## 모범 답안 사용 주의

모범 답안은 하나의 예시다. 학생이 `matplotlib` 만 사용하거나 `seaborn` 만 사용해도 질문에 맞는 차트를 만들고 숫자와 결론이 맞으면 통과시킨다. 다만 차트가 실행만 되고 제목, 축, 결론이 비어 있으면 시각화 과제의 핵심을 놓친 것이므로 보충 지도가 필요하다.

## 재현성 확인

강사는 답안 노트북을 런타임 재시작 후 처음부터 실행해 본다. 시각화 셀은 이전 셀의 변수에 의존하기 쉬우므로 `df`, `daily`, `channel_summary`, `category_summary` 가 순서대로 만들어지는지 확인한다. 여러 그림을 만드는 셀은 `plt.close(fig)` 를 사용하면 검증 로그가 과하게 길어지는 것을 줄일 수 있다.

## 결론 문장 채점 예시

결론이 “매출 그래프를 그렸다”에서 끝나면 부족하다. “전체 매출은 2025-03에 가장 높고, web 채널은 매출 규모가 크지만 app 채널은 ROI가 높아 캠페인 목적에 따라 우선순위를 나누어야 한다”처럼 숫자, 비교, 행동이 함께 들어가야 한다.